In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path('/Users/choedasom/lab_middle_project')
assert (PROJECT_ROOT / 'src' / 'feature.py').is_file(), f'경로 확인 필요: {PROJECT_ROOT}'
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

for m in list(sys.modules):
    if m == 'src' or m.startswith('src.'):
        del sys.modules[m]

from src.feature import add_features

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

final_df = pd.read_parquet(PROCESSED_DIR / 'final_df.parquet')
# 2026-09-02 결정: beta를 ^GSPC(가격지수) 대신 ^SP500TR(총수익지수)로 계산한다.
# 민감도 점검(05번 노트북 참고) 결과 beta 값 자체는 조금만 움직이지만
# (평균절대차 60일 0.0033/120일 0.0019), Ward 2-군집 경계 근처 종목이 많아서
# 안정군 라벨은 60일 84.07%/120일 90.00%만 일치했다 -- "영향 무시 가능"이라고
# 부르기엔 너무 크다. 개별 주가는 이미 배당조정(auto_adjust=True)이라 beta의
# 시장 쪽 수익률도 같은 총수익 정의로 맞추는 게 맞다.
sp500_beta_df = pd.read_parquet(RAW_DIR / 'sp500_total_return_df.parquet')
print(f'final_df: {final_df.shape}, {final_df["Date"].min()} ~ {final_df["Date"].max()}')
print(f'sp500_beta_df (^SP500TR): {sp500_beta_df.shape}, {sp500_beta_df["Date"].min()} ~ {sp500_beta_df["Date"].max()}')

final_df: (1669599, 8), 2016-01-04 00:00:00 ~ 2026-06-30 00:00:00
sp500_beta_df (^SP500TR): (2637, 2), 2016-01-04 00:00:00 ~ 2026-06-30 00:00:00


In [2]:
final_60_df = add_features(final_df, sp500_beta_df=sp500_beta_df, windows=(60,))
final_60_df.to_parquet(PROCESSED_DIR / 'final_60_df.parquet', index=False)
print(f'final_60_df 저장 완료: {final_60_df.shape}, {final_60_df["Date"].min()} ~ {final_60_df["Date"].max()}')
final_60_df.head()

[경고] |일간수익률| > 75% 인 행 121건 / 13종목 → NaN 마스킹
      Date Ticker     Close  daily_return
2024-03-12   SIVB  0.110000     83.615385
2024-03-06   SIVB  0.070000     52.846154
2024-10-04    CHK 81.460000     25.708197
2026-03-13   FRCB  0.003000      6.500000
2025-05-21   FRCB  0.016000      5.400000
2025-08-20   FRCB  0.003100      4.166667
2025-07-09   FRCB  0.005000      4.000000
2026-06-04    STI 22.709999      3.505952
2025-06-10   FRCB  0.010000      3.347826
2026-05-27   FRCB  0.010000      3.081633
        find_price_anomalies()로 원본 가격을 확인하세요 (price_range_ratio > 1000 이면 분할 미조정 의심).


[경고] |return_60d| > 10 인 행 24건 / 4종목 → NaN 마스킹
[요약] 피처별 유효값 비율
beta_60d          0.9733
return_60d        0.9746
rsi_60d           0.9733
volatility_60d    0.9733


final_60_df 저장 완료: (1669599, 12), 2016-01-04 00:00:00 ~ 2026-06-30 00:00:00


,Date,Ticker,Open,High,Low,Close,Volume,source,volatility_60d,beta_60d,rsi_60d,return_60d
0,2016-01-04,A,37.751924,37.871448,37.089932,37.411732,3287300,yahoo,NaN,NaN,NaN,NaN
1,2016-01-05,A,37.448518,37.650795,37.089940,37.283020,2587200,yahoo,NaN,NaN,NaN,NaN
2,2016-01-06,A,36.997993,37.687568,36.823298,37.448513,2103600,yahoo,NaN,NaN,NaN,NaN
3,2016-01-07,A,36.906036,36.915233,35.683192,35.857883,3504300,yahoo,NaN,NaN,NaN,NaN
4,2016-01-08,A,36.060163,36.510683,35.370588,35.480919,3736700,yahoo,NaN,NaN,NaN,NaN


In [3]:
final_120_df = add_features(final_df, sp500_beta_df=sp500_beta_df, windows=(120,))
final_120_df.to_parquet(PROCESSED_DIR / 'final_120_df.parquet', index=False)
print(f'final_120_df 저장 완료: {final_120_df.shape}, {final_120_df["Date"].min()} ~ {final_120_df["Date"].max()}')
final_120_df.head()

[경고] |일간수익률| > 75% 인 행 121건 / 13종목 → NaN 마스킹
      Date Ticker     Close  daily_return
2024-03-12   SIVB  0.110000     83.615385
2024-03-06   SIVB  0.070000     52.846154
2024-10-04    CHK 81.460000     25.708197
2026-03-13   FRCB  0.003000      6.500000
2025-05-21   FRCB  0.016000      5.400000
2025-08-20   FRCB  0.003100      4.166667
2025-07-09   FRCB  0.005000      4.000000
2026-06-04    STI 22.709999      3.505952
2025-06-10   FRCB  0.010000      3.347826
2026-05-27   FRCB  0.010000      3.081633
        find_price_anomalies()로 원본 가격을 확인하세요 (price_range_ratio > 1000 이면 분할 미조정 의심).


[경고] |return_120d| > 10 인 행 121건 / 4종목 → NaN 마스킹
[요약] 피처별 유효값 비율
beta_120d          0.9475
return_120d        0.9492
rsi_120d           0.9475
volatility_120d    0.9475


final_120_df 저장 완료: (1669599, 12), 2016-01-04 00:00:00 ~ 2026-06-30 00:00:00


,Date,Ticker,Open,High,Low,Close,Volume,source,volatility_120d,beta_120d,rsi_120d,return_120d
0,2016-01-04,A,37.751924,37.871448,37.089932,37.411732,3287300,yahoo,NaN,NaN,NaN,NaN
1,2016-01-05,A,37.448518,37.650795,37.089940,37.283020,2587200,yahoo,NaN,NaN,NaN,NaN
2,2016-01-06,A,36.997993,37.687568,36.823298,37.448513,2103600,yahoo,NaN,NaN,NaN,NaN
3,2016-01-07,A,36.906036,36.915233,35.683192,35.857883,3504300,yahoo,NaN,NaN,NaN,NaN
4,2016-01-08,A,36.060163,36.510683,35.370588,35.480919,3736700,yahoo,NaN,NaN,NaN,NaN


In [4]:
# 옛 8개 지표 버전(cagr_10y 포함)으로 되돌아가지 않았는지 확인하는 회귀 가드
assert 'cagr_10y' not in final_60_df.columns
assert 'cagr_10y' not in final_120_df.columns
print('OK: cagr_10y 없음 (4개 지표 버전 확인)')

OK: cagr_10y 없음 (4개 지표 버전 확인)


In [5]:
cluster_60df = final_60_df[['Date', 'Ticker', 'beta_60d', 'volatility_60d', 'return_60d', 'rsi_60d']].copy()
print(cluster_60df.shape)
cluster_60df.head()

(1669599, 6)


,Date,Ticker,beta_60d,volatility_60d,return_60d,rsi_60d
0,2016-01-04,A,NaN,NaN,NaN,NaN
1,2016-01-05,A,NaN,NaN,NaN,NaN
2,2016-01-06,A,NaN,NaN,NaN,NaN
3,2016-01-07,A,NaN,NaN,NaN,NaN
4,2016-01-08,A,NaN,NaN,NaN,NaN


In [6]:
cluster_120df = final_120_df[['Date', 'Ticker', 'beta_120d', 'volatility_120d', 'return_120d', 'rsi_120d']].copy()
print(cluster_120df.shape)


(1669599, 6)


In [7]:
cluster_60df.to_parquet(PROCESSED_DIR / 'cluster_60df.parquet', index=False)
cluster_120df.to_parquet(PROCESSED_DIR / 'cluster_120df.parquet', index=False)
print(f'cluster_60df 저장 완료: {PROCESSED_DIR / "cluster_60df.parquet"}')
print(f'cluster_120df 저장 완료: {PROCESSED_DIR / "cluster_120df.parquet"}')

cluster_60df 저장 완료: /Users/choedasom/lab_middle_project/data/processed/cluster_60df.parquet
cluster_120df 저장 완료: /Users/choedasom/lab_middle_project/data/processed/cluster_120df.parquet


# 리밸런싱이 왜 필요한가

특정 날짜 하나만 골라서 군집화하면, "그 순간 안정적인 종목"을 딱 한 번 뽑는 셈이라 **표본이 1개**밖에 안 된다. 그 선택이 좋은 성과로 이어져도, 전략이 진짜 좋은 건지 그냥 운이 좋았던 건지 구분할 수 없다.

그래서 60일마다 시점을 바꿔가며 반복적으로 다시 판단(=리밸런싱)한다. 10년치 데이터를 60일 간격으로 나누면 약 43번의 독립적인 판단 시점이 생기고, 이 43번의 결과를 모아봐야 "이 전략을 꾸준히 실행했을 때 실제로 통했는가"를 통계적으로 의미 있게 검증할 수 있다.

In [8]:
# 매일 다 보면 표본이 너무 많고 서로 겹쳐서(어제와 오늘은 거의 같은 값) 의미가 약하다.
# 60일 간격으로 끊어서, 서로 겹치지 않는 독립적인 판단 시점 43개를 뽑는다.
all_dates = sorted(cluster_60df['Date'].unique())
rebalance_dates = all_dates[::60]
print(f'전체 거래일: {len(all_dates)}, 리밸런싱 시점: {len(rebalance_dates)}개')
print(f'첫 시점: {rebalance_dates[0]}, 마지막 시점: {rebalance_dates[-1]}')

# 리밸런싱 시점에 해당하는 행만 남긴다 (= 매 60일마다 딱 한 번씩 포트폴리오를 다시 짜는 시뮬레이션)
rebalance_snapshots = cluster_60df[cluster_60df['Date'].isin(rebalance_dates)].copy()
print(f'스냅샷 shape: {rebalance_snapshots.shape}')
rebalance_snapshots.head()

전체 거래일: 2637, 리밸런싱 시점: 44개
첫 시점: 2016-01-04 00:00:00, 마지막 시점: 2026-04-09 00:00:00
스냅샷 shape: (27867, 6)


,Date,Ticker,beta_60d,volatility_60d,return_60d,rsi_60d
0,2016-01-04,A,NaN,NaN,NaN,NaN
60,2016-03-31,A,1.271214,0.285565,-0.020644,49.314015
120,2016-06-24,A,1.239632,0.203317,0.110356,60.183147
180,2016-09-20,A,1.559464,0.226525,0.038768,53.663637
240,2016-12-14,A,1.342155,0.228246,0.011914,51.464840


# 스냅샷이 다음 리밸런싱까지 얼마나 어긋나는지 측정

리밸런싱 시점에 찍은 값(예: `beta_60d`)이 다음 리밸런싱 시점(60일 뒤)엔 얼마나 달라져 있는지, 종목별로 직접 비교한다. 차이가 크면 "스냅샷 하나로 60일을 대표한다"는 가정이 위험하다는 뜻이고, 작으면 비교적 안전하다는 뜻이다.

In [9]:
feature_cols = ['beta_60d', 'volatility_60d', 'return_60d', 'rsi_60d']

# 종목별로 시간순 정렬한 뒤, 바로 이전 리밸런싱 값과의 차이를 구한다 (diff = 이번 값 - 직전 값)
drift = rebalance_snapshots.sort_values(['Ticker', 'Date']).copy()
for col in feature_cols:
    drift[f'{col}_drift'] = drift.groupby('Ticker')[col].diff()

drift_cols = [f'{col}_drift' for col in feature_cols]
drift[drift_cols].describe()

,beta_60d_drift,volatility_60d_drift,return_60d_drift,rsi_60d_drift
count,26408.000000,26408.000000,26454.000000,26408.000000
mean,-0.007365,0.000547,-0.000916,-0.058816
std,0.523321,0.171462,0.250430,11.372728
min,-4.149043,-2.197498,-4.954545,-91.133348
25%,-0.273971,-0.063531,-0.126108,-7.906771
50%,-0.007327,-0.003007,-0.005517,-0.040262
75%,0.262641,0.052969,0.116566,7.686529
max,5.188889,2.756760,5.333333,65.561354


In [10]:
# drift의 표준편차를 원본 값의 표준편차로 나눠서, "원래 변동폭 대비 얼마나 크게 어긋나는지" 비율로 비교
for col in feature_cols:
    original_std = rebalance_snapshots[col].std()
    drift_std = drift[f'{col}_drift'].std()
    print(f'{col}: 원본 표준편차={original_std:.4f}, drift 표준편차={drift_std:.4f}, 비율={drift_std/original_std:.2f}')

beta_60d: 원본 표준편차=0.5833, drift 표준편차=0.5233, 비율=0.90
volatility_60d: 원본 표준편차=0.1858, drift 표준편차=0.1715, 비율=0.92
return_60d: 원본 표준편차=0.1766, drift 표준편차=0.2504, 비율=1.42
rsi_60d: 원본 표준편차=8.0866, drift 표준편차=11.3727, 비율=1.41


# 알려진 한계: 지표 drift (2026-08-26 결정)

`return_60d`(비율 1.40), `rsi_60d`(비율 1.24)는 리밸런싱 주기(60일) 안에서도 종목 간 차이보다 더 크게 흔들릴 수 있다는 게 위에서 확인됨. `beta_60d`(0.84), `volatility_60d`(0.99)는 상대적으로 덜 흔들림.

**결정**: 지금 구조(지표별 리밸런싱 주기 통일, 60일)를 그대로 유지하고 43번 백테스팅을 먼저 끝까지 돌린다. 지표마다 주기를 다르게 하는 등의 구조 변경은 하지 않는다 — 아직 실제로 성과에 문제가 되는지 확인되지 않은 상태에서 구조를 복잡하게 만들면, 나중에 결과가 나빠졌을 때 원인이 drift 때문인지 다른 요인 때문인지 구분하기 어려워지기 때문이다.

**나중에 다시 볼 것**: 백테스팅 결과가 기대에 못 미치면, 이 drift(특히 `return_60d`/`rsi_60d`)를 원인 후보로 먼저 검토한다.

In [11]:
rebalance_60df = rebalance_snapshots.copy()
rebalance_60df

,Date,Ticker,beta_60d,volatility_60d,return_60d,rsi_60d
0,2016-01-04,A,NaN,NaN,NaN,NaN
60,2016-03-31,A,1.271214,0.285565,-0.020644,49.314015
120,2016-06-24,A,1.239632,0.203317,0.110356,60.183147
180,2016-09-20,A,1.559464,0.226525,0.038768,53.663637
240,2016-12-14,A,1.342155,0.228246,0.011914,51.464840
...,...,...,...,...,...,...
1669302,2025-04-24,ZTS,0.559103,0.326089,-0.103310,44.735097
1669362,2025-07-22,ZTS,1.019018,0.274147,-0.003969,50.326529
1669422,2025-10-15,ZTS,0.588649,0.191146,-0.059991,44.940954
1669482,2026-01-12,ZTS,1.141926,0.365011,-0.115140,42.967138


# 120일 기준 리밸런싱

`cluster_120df`는 지표 자체가 120일 롤링으로 계산돼 있으니, 리밸런싱 간격도 지표 윈도우와 맞춰서 120일로 맞춘다 (60일 리밸런싱과 동일한 로직, 간격만 다름).

In [12]:
all_dates_120 = sorted(cluster_120df['Date'].unique())
rebalance_dates_120 = all_dates_120[::120]
print(f'전체 거래일: {len(all_dates_120)}, 리밸런싱 시점: {len(rebalance_dates_120)}개')
print(f'첫 시점: {rebalance_dates_120[0]}, 마지막 시점: {rebalance_dates_120[-1]}')

rebalance_snapshots_120 = cluster_120df[cluster_120df['Date'].isin(rebalance_dates_120)].copy()
print(f'스냅샷 shape: {rebalance_snapshots_120.shape}')
rebalance_snapshots_120.head()

전체 거래일: 2637, 리밸런싱 시점: 22개
첫 시점: 2016-01-04 00:00:00, 마지막 시점: 2026-01-12 00:00:00
스냅샷 shape: (13936, 6)


,Date,Ticker,beta_120d,volatility_120d,return_120d,rsi_120d
0,2016-01-04,A,NaN,NaN,NaN,NaN
120,2016-06-24,A,1.257015,0.247354,0.087434,53.613902
240,2016-12-14,A,1.469323,0.226456,0.051144,52.552879
360,2017-06-08,A,1.478515,0.176875,0.310661,64.156458
480,2017-11-28,A,1.196553,0.149369,0.178603,60.050788


In [13]:
feature_cols_120 = ['beta_120d', 'volatility_120d', 'return_120d', 'rsi_120d']

drift_120 = rebalance_snapshots_120.sort_values(['Ticker', 'Date']).copy()
for col in feature_cols_120:
    drift_120[f'{col}_drift'] = drift_120.groupby('Ticker')[col].diff()

for col in feature_cols_120:
    original_std = rebalance_snapshots_120[col].std()
    drift_std = drift_120[f'{col}_drift'].std()
    print(f'{col}: 원본 표준편차={original_std:.4f}, drift 표준편차={drift_std:.4f}, 비율={drift_std/original_std:.2f}')

beta_120d: 원본 표준편차=0.5030, drift 표준편차=0.3805, 비율=0.76
volatility_120d: 원본 표준편차=0.1676, drift 표준편차=0.1397, 비율=0.83
return_120d: 원본 표준편차=0.2767, drift 표준편차=0.3662, 비율=1.32
rsi_120d: 원본 표준편차=5.7718, drift 표준편차=8.0491, 비율=1.39


## 알려진 한계: 지표 drift (2026-08-26 결정, 2026-08-29 데이터 정제 후 재검증)
60일: beta 0.90 / volatility 0.93 (안정) / return 1.41 / rsi 1.41 (불안정)
120일: beta 0.76 / volatility 0.84 (안정) / return 1.32 / rsi 1.40 (불안정)
결론 방향은 최초 결정과 동일하게 유지됨. 120일 beta가 경계선(1.07)에서
명확한 안정(0.76)으로 이동 — 데이터 오염(PARA 등)이 이 지표를 왜곡하고 있었음을 시사.

In [14]:
rebalance_120df=rebalance_snapshots_120.copy()
rebalance_120df

,Date,Ticker,beta_120d,volatility_120d,return_120d,rsi_120d
0,2016-01-04,A,NaN,NaN,NaN,NaN
120,2016-06-24,A,1.257015,0.247354,0.087434,53.613902
240,2016-12-14,A,1.469323,0.226456,0.051144,52.552879
360,2017-06-08,A,1.478515,0.176875,0.310661,64.156458
480,2017-11-28,A,1.196553,0.149369,0.178603,60.050788
...,...,...,...,...,...,...
1669002,2024-02-12,ZTS,1.046935,0.231560,0.092905,53.860933
1669122,2024-08-05,ZTS,0.833403,0.283741,-0.106536,46.879465
1669242,2025-01-28,ZTS,0.313347,0.218237,-0.019905,49.636956
1669362,2025-07-22,ZTS,0.611513,0.300273,-0.106869,47.243315


In [15]:
rebalance_60df.to_parquet(PROCESSED_DIR / 'rebalance_60df.parquet', index=False)
rebalance_120df.to_parquet(PROCESSED_DIR / 'rebalance_120df.parquet', index=False)
print(f'rebalance_60df 저장 완료: {PROCESSED_DIR / "rebalance_60df.parquet"}, shape={rebalance_60df.shape}')
print(f'rebalance_120df 저장 완료: {PROCESSED_DIR / "rebalance_120df.parquet"}, shape={rebalance_120df.shape}')

rebalance_60df 저장 완료: /Users/choedasom/lab_middle_project/data/processed/rebalance_60df.parquet, shape=(27867, 6)
rebalance_120df 저장 완료: /Users/choedasom/lab_middle_project/data/processed/rebalance_120df.parquet, shape=(13936, 6)


# 리밸런싱 시점별 자격(point-in-time membership) 필터

지금까지의 `rebalance_60df`/`rebalance_120df`는 "그 날짜에 가격 데이터가 있으면" 무조건 후보로 넣었다.
그런데 가격 데이터가 있다는 것과 "그 시점에 실제 S&P500 구성종목이었다"는 것은 다르다 — 예를 들어
`MRVL`은 2026-06-22에야 편입됐는데, 2016년 스냅샷에도 후보로 잘못 들어가 있었다(실제 검증 결과
27,757개 중 6,208개, 22.4%가 이런 경우였다). `src/get_tickers.py`의 `filter_by_membership()`으로
각 (종목, 날짜) 조합이 실제로 그 시점의 구성종목이었는지 확인해서 걸러낸다.

In [16]:
from src.get_tickers import filter_by_membership

print('=== 60일 리밸런싱 ===')
rebalance_60df = filter_by_membership(rebalance_60df)
print('필터 후 shape:', rebalance_60df.shape, '| 종목 수:', rebalance_60df['Ticker'].nunique())

print()
print('=== 120일 리밸런싱 ===')
rebalance_120df = filter_by_membership(rebalance_120df)
print('필터 후 shape:', rebalance_120df.shape, '| 종목 수:', rebalance_120df['Ticker'].nunique())

rebalance_60df.to_parquet(PROCESSED_DIR / 'rebalance_60df.parquet', index=False)
rebalance_120df.to_parquet(PROCESSED_DIR / 'rebalance_120df.parquet', index=False)
print()
print('시점별 자격 필터 반영해서 재저장 완료')

=== 60일 리밸런싱 ===
[캐시] 편입/편출 이력 1259건 재사용 (/Users/choedasom/lab_middle_project/data/raw/cache/sp500_membership_history.csv)
[INFO] 시점별 자격 미달로 제외된 (종목,날짜) 조합: 6098건 / 27867건
필터 후 shape: (21769, 6) | 종목 수: 683

=== 120일 리밸런싱 ===
[캐시] 편입/편출 이력 1259건 재사용 (/Users/choedasom/lab_middle_project/data/raw/cache/sp500_membership_history.csv)
[INFO] 시점별 자격 미달로 제외된 (종목,날짜) 조합: 3061건 / 13936건
필터 후 shape: (10875, 6) | 종목 수: 675

시점별 자격 필터 반영해서 재저장 완료


# 알려진 한계: 사명 변경(rename) 검증 범위 (2026-09-01 결정)

`TRUSTED_RENAMES`(`src/get_tickers.py`)는 15개 티커 변경 쌍을 수동으로 등록해둔 것이다.
이걸 어디까지, 어떻게 검증했는지 정리한다.

**검증한 것**
1. 15개 매핑 각각 — raw 멤버십 이력에서 옛 티커의 마지막 `end_date`와 새 티커의 첫
   `start_date`가 정확히 맞물리는지 확인. 13/15는 완전히 일치. 나머지 2개(`FBHS→FBIN`,
   `GPS→GAP`)는 안 맞물려서 뉴스로 직접 확인함 — 둘 다 실제로는 무해했음
   (FBIN은 애초에 S&P500 멤버였던 적이 없고, GPS→GAP은 편출과 무관한 시점에 일어난
   티커 변경이지만 가격 데이터가 과거 구간까지 전부 새 티커로 기록돼 있어서 매핑이
   꼭 필요했음).
2. `final_df`에 실제로 가격 데이터가 있는 707개 티커 전부가, `TRUSTED_RENAMES` 적용 후
   멤버십 이력에 최소 한 번은 매칭되는지 확인 — **누락(orphan) 0건.** 가격 데이터가 있는
   종목이 멤버십 데이터 불일치로 통째로 조용히 탈락하는 경우는 없다는 뜻.
3. 멤버십 구간 자체의 구조적 오류(같은 티커 내 구간 겹침, 30일 미만 극단적으로 짧은 구간)
   — final_df 관련 764건 기준으로 없음.

**검증 못 한 것 / 왜 포기했는지**
- raw 멤버십 이력 전체 1,259건 중 뉴스로 직접 확인한 건 리네임 관련 30건 안팎뿐이고,
  나머지(97~98%)는 `fja05680/sp500` GitHub 저장소를 그대로 신뢰하는 중이다.
- "경계 날짜가 정확히 맞물리는 쌍"을 자동 탐지해서 놓친 리네임을 찾으려 했으나
  (`find_rename_candidates.py`), 1,105건의 후보가 나왔고 그중 무관한 종목이 같은 날
  우연히 교체된 경우(예: `BSC`→`ISRG`, 베어스턴스 파산일에 인튜이티브 서지컬 편입)와
  진짜 리네임이 데이터상 구별이 안 됨. final_df 관련 티커로 좁혀도 820건이나 남아
  사람이 하나하나 확인해야 하는 규모를 못 벗어남 — 이 방향은 포기.

**결정**: 위 2번(가격 데이터 707개 전부 매칭 확인)이 이 프로젝트에 실제 영향을 주는
유일한 검증이라고 판단하고, 여기서 멈춘다. raw 이력 전체의 정확성은 감사(audit)하지
않은 상태로 남겨둔다 — CIK 같은 독립 식별자 기반 검증을 붙이지 않는 한, 날짜 패턴
만으로는 이 이상 좁혀지지 않는다.